# Content

### Summary of the Project

This project builds an RNN-based sentiment analysis model to classify movie reviews as positive or negative.

1. **Objective**:  
   To determine whether a given movie review is positive or negative based on its text content.

2. **Model Used**:  
   - **Recurrent Neural Network (RNN)**: The model uses an embedding layer followed by two LSTM (Long Short-Term Memory) layers to capture sequential dependencies in text data.
   - A dropout layer is added to prevent overfitting, and two dense layers (one with a ReLU activation and the other with a sigmoid activation) are used for classification.

3. **Data**:  
   - The dataset contains movie reviews for training and testing. Training data includes sentiment labels, while test data contains only reviews without labels.
   - Preprocessing steps include removing HTML tags, converting text to lowercase, removing special characters, and removing stopwords.

4. **Approach**:  
   - The text is tokenized and converted into padded sequences to ensure uniform input size.
   - The model is trained using the training data, and predictions are made on the test data.
   - Sample predictions are displayed with the corresponding reviews.

5. **Prediction Output**:  
   - The model predicts the sentiment (positive or negative) for several sample reviews from the test dataset. Results are printed along with excerpts of the reviews for interpretation.

6. **Evaluation**:  
   - Since test data lacks sentiment labels, predictions are presented without a quantitative evaluation metric.

This approach demonstrates the ability to preprocess text data and build an RNN model for binary sentiment classification.

# Basic setting

### Measuring execution time

In [103]:
!pip install --q ipython-autotime
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 2.83 s (started: 2025-01-24 03:00:41 +00:00)


### Library

In [104]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
from keras import layers
import os
import requests
import zipfile
import io

from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.metrics import classification_report, confusion_matrix


time: 1.23 ms (started: 2025-01-24 03:00:44 +00:00)


# Data collection

### Load data with zip file from URL

In [105]:
# Zip file URL
zip_url = 'https://raw.githubusercontent.com/20161609/data_box/main/movie_review.zip'
response = requests.get(zip_url)
if response.status_code == 200:
    zip_data = io.BytesIO(response.content)  # Processing in memory
    with zipfile.ZipFile(zip_data, 'r') as zip_ref:
        zip_ref.extractall('/content')  # Decompress in memory

    data_root = '/content/movie_review'
    train_dir = data_root + '/train'
    test_dir = data_root + '/test'
else:
    raise Exception(f"Failed to download data. Status code: {response.status_code}")

train = pd.read_csv('/content/trainData.tsv', delimiter='\t')
test = pd.read_csv('/content/testData.tsv', delimiter='\t')

train.shape, test.shape

((25000, 3), (25000, 2))

time: 8.79 s (started: 2025-01-24 03:00:44 +00:00)


In [106]:
train.head()

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...


time: 438 ms (started: 2025-01-24 03:00:53 +00:00)


In [107]:
train.iloc[0].review[:40:]

'With all this stuff going down at the mo'

time: 5.6 ms (started: 2025-01-24 03:00:53 +00:00)


# Data preprocessing

### Delete HTML tag

In [108]:
from bs4 import BeautifulSoup

bs = BeautifulSoup(train.iloc[0].review, 'html.parser')
bs.text

"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.The actual feature film bit when it finally starts is only on for 20 mi

time: 32.8 ms (started: 2025-01-24 03:00:53 +00:00)


### Delete numbers and symbols - regular expression

In [109]:
import re

cleaned = re.sub('[^a-zA-Z]', ' ', bs.text)
cleaned

'With all this stuff going down at the moment with MJ i ve started listening to his music  watching the odd documentary here and there  watched The Wiz and watched Moonwalker again  Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent  Moonwalker is part biography  part feature film which i remember going to see at the cinema when it was originally released  Some of it has subtle messages about MJ s feeling towards the press and also the obvious message of drugs are bad m kay Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring  Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him The actual feature film bit when it finally starts is only on for    mi

time: 14.1 ms (started: 2025-01-24 03:00:54 +00:00)


In [110]:
### Uppercase -> Lowercase

time: 296 µs (started: 2025-01-24 03:00:54 +00:00)


In [111]:
leaned = cleaned.lower()
cleaned

'With all this stuff going down at the moment with MJ i ve started listening to his music  watching the odd documentary here and there  watched The Wiz and watched Moonwalker again  Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent  Moonwalker is part biography  part feature film which i remember going to see at the cinema when it was originally released  Some of it has subtle messages about MJ s feeling towards the press and also the obvious message of drugs are bad m kay Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring  Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him The actual feature film bit when it finally starts is only on for    mi

time: 7.91 ms (started: 2025-01-24 03:00:54 +00:00)


### stopwords

In [112]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

time: 104 ms (started: 2025-01-24 03:00:54 +00:00)


### Retrieve stopwords.

In [113]:
from nltk.corpus import stopwords

eng_stopwords = stopwords.words('english')
eng_stopwords[:5:]

['i', 'me', 'my', 'myself', 'we']

time: 7.71 ms (started: 2025-01-24 03:00:54 +00:00)


### Text preprocessing

In [114]:
def preprocess(sentence):
    soup = BeautifulSoup(sentence, 'html.parser')
    cleaned = re.sub('[^a-zA-Z]', ' ', soup.text)
    cleaned = cleaned.lower()
    cleaned = [word for word in cleaned.split() if word not in eng_stopwords]
    return ' '.join(cleaned)

preprocess(train.iloc[0].review)

train_clean = train['review'].apply(preprocess)
train_clean.head()

<ipython-input-114-2ba130426d2b>:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(sentence, 'html.parser')


,review
0,stuff going moment mj started listening music ...
1,classic war worlds timothy hines entertaining ...
2,film starts manager nicholas bell giving welco...
3,must assumed praised film greatest filmed oper...
4,superbly trashy wondrously unpretentious explo...


time: 35.2 s (started: 2025-01-24 03:00:54 +00:00)


# Data preparation

## Tokenizer

In [115]:
tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(train_clean)

len(tokenizer.word_index), tokenizer.word_index['odd']

(74066, 874)

time: 3.05 s (started: 2025-01-24 03:01:29 +00:00)


### Split: train and test

In [116]:
train_lable = train['sentiment']
train_lable.head()

,sentiment
0,1
1,1
2,0
3,0
4,1


time: 7.32 ms (started: 2025-01-24 03:01:32 +00:00)


In [117]:
from sklearn.model_selection import train_test_split

train_sentence, val_sentence, train_label, val_label = train_test_split(train_clean, train_lable, test_size=0.2, random_state=42)
train_sentence.shape, val_sentence.shape

((20000,), (5000,))

time: 9.99 ms (started: 2025-01-24 03:01:32 +00:00)


### Sequence

In [118]:
train_sequence = tokenizer.texts_to_sequences(train_sentence)
val_sequence = tokenizer.texts_to_sequences(val_sentence)

print(train_sequence[0])

[2, 887, 842, 821, 3003, 14250, 1710, 3772, 24779, 1175, 3, 3297, 1485, 8391, 1710, 3772, 4, 1086, 1052, 168, 26856, 869, 29536, 7405, 160, 1167, 177, 3134, 589, 3772, 1861, 80, 819, 15, 726, 3, 375, 3772, 10390, 3226, 16, 357, 13, 842, 15, 134, 124, 4, 713, 1160, 16758, 3750, 1110, 398, 2, 99, 173, 4304, 178, 155, 72211, 685, 72212, 1673, 16, 118, 2316, 8277, 1200, 53, 2209, 1861, 7592, 34495, 72213, 482, 72214, 1144, 246, 2517, 2191, 2808, 22, 366, 246, 2497, 53, 6810, 205, 138, 839, 87, 17142, 3750, 1110, 16, 118, 6049, 3772, 3911, 4265, 1411, 450, 415, 4265, 172, 1374, 122, 363, 118, 1094, 112, 246, 2265, 1364, 1580, 1797, 16, 101, 32, 170, 106, 30, 614]
time: 1.21 s (started: 2025-01-24 03:01:32 +00:00)


### Pedding

In [119]:
from keras.preprocessing.sequence import pad_sequences

train_padded = pad_sequences(train_sequence,
              maxlen=150,
              padding='pre',
              truncating='pre')

val_padded = pad_sequences(val_sequence,
              maxlen=150,
              padding='pre',
              truncating='pre')

train_padded.shape

(20000, 150)

time: 204 ms (started: 2025-01-24 03:01:33 +00:00)


# Model

### Create model

In [120]:
train_label = train_label.to_numpy()
val_label = val_label.to_numpy()
train_label

EMBEDDING_DIM = 300
VAOCA_SIZE = len(tokenizer.word_index)+1
VAOCA_SIZE

model = keras.Sequential([
    layers.Embedding(VAOCA_SIZE, EMBEDDING_DIM, input_length=150),
    layers.LSTM(128, return_sequences=True),
    layers.LSTM(128),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_4 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_5 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

time: 39.8 ms (started: 2025-01-24 03:01:33 +00:00)


### Compile model

In [121]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

time: 8.27 ms (started: 2025-01-24 03:01:34 +00:00)


### Learning and get history

In [122]:
EPOCHS = 1
BATCH_SIZE = 32

history = model.fit(
    train_padded, train_label,
    epochs = EPOCHS,
    batch_size = BATCH_SIZE,
    # validation_split = 0.2
    validation_data=(val_padded, val_label)
)

625/625 ━━━━━━━━━━━━━━━━━━━━ 625s 991ms/step - accuracy: 0.7594 - loss: 0.4892 - val_accuracy: 0.8752 - val_loss: 0.3360
time: 10min 25s (started: 2025-01-24 03:01:34 +00:00)


# Estimate

### Test data

In [123]:
# Preprocess test data
test_clean = test['review'].apply(preprocess)
test_sequence = tokenizer.texts_to_sequences(test_clean)
test_padded = pad_sequences(test_sequence, maxlen=150, padding='pre', truncating='pre')

# Predict on test data
predictions = model.predict(test_padded, batch_size=32)
predictions = [1 if p > 0.5 else 0 for p in predictions]  # Convert probabilities to binary values

# Select a few samples to display
sample_indices = [0, 10, 20, 30, 40]  # Adjust indices as needed
for idx in sample_indices:
    print(f"Review: {test.iloc[idx].review[:100]}...")  # Display first 100 characters of the review
    print(f"Predicted Sentiment: {'Positive' if predictions[idx] == 1 else 'Negative'}")
    print("-" * 80)


<ipython-input-114-2ba130426d2b>:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(sentence, 'html.parser')


782/782 ━━━━━━━━━━━━━━━━━━━━ 166s 212ms/step
Review: Naturally in a film who's main themes are of mortality, nostalgia, and loss of innocence it is perha...
Predicted Sentiment: Positive
--------------------------------------------------------------------------------
Review: 'The Mill on the Floss' was one of the lesser novels by Mary Ann Evans, who wrote under the male pse...
Predicted Sentiment: Negative
--------------------------------------------------------------------------------
Review: I like action movies. I have a softspot for \B\" flicks with bad dialogue and wooden acting. So, I'v...
Predicted Sentiment: Negative
--------------------------------------------------------------------------------
Review: If I was only allowed to watch one program in my entire life, I would definitely have to pick \The C...
Predicted Sentiment: Positive
--------------------------------------------------------------------------------
Review: I saw it tonight and fell asleep in the movie.<br /><br